# 06_tool_annotations

Stage 6 of the `codes/mcp/` track: **tool annotations + `_meta`** on the same
**FastMCP + Pydantic** stack as stages 1–5.

Notebook path: `codes/mcp/06_tool_annotations/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Install once (venv recommended), from this stage directory:

```bash
pip install -r ../../requirements.txt
```

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`@mcp.tool(annotations=..., meta=...)` → `tools/list` carries `ToolAnnotations`
behavior hints + free-form `_meta` → host gates auto-approve / confirm / retry
without guessing from the tool name alone.


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "06_tool_annotations" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "06_tool_annotations"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/06_tool_annotations")

ROOT = lab_root()
print(ROOT)


/workspace/AEFS-Notes/codes/mcp/06_tool_annotations


## 1. Read a host-facing tools/list annotation shape

`payload` is a trimmed `tools/list` entry. Print the tool `name`, the
`annotations.readOnlyHint`, and `annotations.destructiveHint`. Confirm a
read-only tool is not marked destructive.


In [2]:
payload = {
    "name": "peek_note",
    "title": "Peek Note",
    "annotations": {
        "title": "Peek Note",
        "readOnlyHint": True,
        "destructiveHint": False,
        "idempotentHint": True,
        "openWorldHint": False,
    },
    "_meta": {"category": "notes", "risk": "low"},
}

# answer

name = payload["name"]
read_only = payload["annotations"]["readOnlyHint"]
destructive = payload["annotations"]["destructiveHint"]
print(name)
print(read_only)
print(destructive)
assert name == "peek_note"
assert read_only is True
assert destructive is False
assert read_only and not destructive


peek_note
True
False


## 2. Tiny FastMCP server with annotated tools

Create a local `FastMCP("qa-ann")` with two tools:

1. `status() -> str` returning `"ok"`, with
   `annotations=ToolAnnotations(title="Status", read_only_hint=True,
   destructive_hint=False, idempotent_hint=True, open_world_hint=False)`
   and `meta={"risk": "low"}`
2. `reset(confirm: bool = False) -> str` returning `"reset"` when
   `confirm` else `"noop"`, with dict annotations
   `{"title": "Reset", "read_only_hint": False, "destructive_hint": True,
   "idempotent_hint": True, "open_world_hint": False}` and
   `meta={"risk": "high"}`

Drive it with `Client(server)`. Print:

1. negotiated `protocol_version` (expect `2026-07-28`)
2. for each tool: `name`, `annotations.read_only_hint`,
   `annotations.destructive_hint`, and `meta["risk"]`


In [3]:
from typing import Annotated

from fastmcp import Client, FastMCP
from mcp.types import ToolAnnotations
from pydantic import Field

# answer

server = FastMCP("qa-ann")

@server.tool(
    annotations=ToolAnnotations(
        title="Status",
        read_only_hint=True,
        destructive_hint=False,
        idempotent_hint=True,
        open_world_hint=False,
    ),
    meta={"risk": "low"},
)
def status() -> str:
    """Safe health check."""
    return "ok"

@server.tool(
    annotations={
        "title": "Reset",
        "read_only_hint": False,
        "destructive_hint": True,
        "idempotent_hint": True,
        "open_world_hint": False,
    },
    meta={"risk": "high"},
)
def reset(confirm: Annotated[bool, Field(description="Must be true")] = False) -> str:
    """Destructive reset."""
    return "reset" if confirm else "noop"

async with Client(server) as client:
    print("protocol:", client.protocol_version)
    tools = await client.list_tools()
    for t in tools:
        ann = t.annotations
        print(t.name, ann.read_only_hint, ann.destructive_hint, t.meta["risk"])
    by_name = {t.name: t for t in tools}
    assert client.protocol_version == "2026-07-28"
    assert by_name["status"].annotations.read_only_hint is True
    assert by_name["reset"].annotations.destructive_hint is True
    assert by_name["status"].meta["risk"] == "low"
    assert by_name["reset"].meta["risk"] == "high"


protocol: 2026-07-28
status True False low
reset False True high


## 3. Drive the lab server: list hints, then call

Import `mcp` from `server.py` in this directory. With `Client(mcp)`:

1. print `protocol_version` and `server_info.name`
2. list tools; for each print `name`, `annotations.title`,
   `read_only_hint`, `destructive_hint`, and `meta["risk"]`
3. call `peek_note` with `{"slug": "alpha"}` — print the text content
4. call `wipe_note` with `{"slug": "alpha", "confirm": False}` — expect
   `"noop"` (destructive tool still gated by its own confirm arg)

Same Client entry as earlier stages — only the `tools/list` surface grows
to carry annotations / `_meta`.


In [4]:
import sys

sys.path.insert(0, str(ROOT))
from fastmcp import Client
from server import mcp

# answer

async with Client(mcp) as client:
    print("protocol:", client.protocol_version)
    print("server:", client.server_info.name if client.server_info else None)
    tools = await client.list_tools()
    for t in tools:
        a = t.annotations
        print(t.name, a.title, a.read_only_hint, a.destructive_hint, t.meta["risk"])

    peeked = await client.call_tool("peek_note", {"slug": "alpha"})
    print("peek:", peeked.content[0].text)

    gated = await client.call_tool("wipe_note", {"slug": "alpha", "confirm": False})
    print("wipe(confirm=False):", gated.content[0].text)

    assert client.protocol_version == "2026-07-28"
    assert client.server_info.name == "lab-tool-annotations"
    names = {t.name for t in tools}
    assert names == {"peek_note", "append_note", "wipe_note"}
    by = {t.name: t for t in tools}
    assert by["peek_note"].annotations.read_only_hint is True
    assert by["wipe_note"].annotations.destructive_hint is True
    assert peeked.content[0].text == "first note"
    assert gated.content[0].text == "noop"


protocol: 2026-07-28
server: lab-tool-annotations
peek_note Peek Note True False low
append_note Append Note False False medium
wipe_note Wipe Note False True high


peek: first note
wipe(confirm=False): noop


## 4. Host-side filter: auto-approve only read-only tools

Build a local `FastMCP("qa-filter")` with `safe() -> str` (`read_only_hint=True`)
and `danger() -> str` (`destructive_hint=True`, `read_only_hint=False`).
List tools and print the names where `annotations.read_only_hint` is True —
expect only `["safe"]`.

Extension point: hosts gate UX from advertised hints; they do not infer
safety from the tool name string.


In [5]:
# answer

filt = FastMCP("qa-filter")

@filt.tool(
    annotations=ToolAnnotations(
        title="Safe",
        read_only_hint=True,
        destructive_hint=False,
        idempotent_hint=True,
        open_world_hint=False,
    ),
)
def safe() -> str:
    return "safe"

@filt.tool(
    annotations=ToolAnnotations(
        title="Danger",
        read_only_hint=False,
        destructive_hint=True,
        idempotent_hint=False,
        open_world_hint=False,
    ),
)
def danger() -> str:
    return "danger"

async with Client(filt) as client:
    tools = await client.list_tools()
    auto = [t.name for t in tools if t.annotations and t.annotations.read_only_hint]
    print(auto)
    assert auto == ["safe"]


['safe']
